# Chapter 03: Conditional Filtering

One of the most frequent operations in data analysis is selecting rows that meet certain criteria. Pandas provides several expressive ways to filter data based on conditions.

In this notebook we will cover:

- Boolean conditions on columns
- Filtering rows with single and multiple conditions
- The `&`, `|`, `~` operators (with parentheses)
- `.isin()` and `.between()`
- The `.query()` method
- Applying filters and assigning results

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
tips = pd.read_csv('data/tips.csv')
tips.head()

## Boolean Conditions

Applying a comparison operator to a column produces a boolean Series — `True` where the condition is met and `False` otherwise.

In [ ]:
# Which bills are over $30?
tips['total_bill'] > 30

In [ ]:
# Use the boolean Series to filter rows
big_bills = tips[tips['total_bill'] > 30]
big_bills.head()

In [ ]:
print(f"Total rows: {len(tips)}, Bills over $30: {len(big_bills)}")

## Filtering on String Columns

In [ ]:
# Select only dinner meals
dinner = tips[tips['time'] == 'Dinner']
dinner.head()

In [ ]:
# Rows where the day is NOT Saturday
not_saturday = tips[tips['day'] != 'Sat']
not_saturday['day'].value_counts()

## Multiple Conditions

When combining conditions, you **must**:

1. Wrap each condition in **parentheses**
2. Use bitwise operators: `&` (and), `|` (or), `~` (not)

Python's `and` / `or` / `not` keywords will **not** work because they cannot operate on arrays.

In [ ]:
# AND: Dinner AND bill over $20
expensive_dinner = tips[(tips['time'] == 'Dinner') & (tips['total_bill'] > 20)]
print(f"Expensive dinners: {len(expensive_dinner)} rows")
expensive_dinner.head()

In [ ]:
# OR: Saturday OR Sunday
weekend = tips[(tips['day'] == 'Sat') | (tips['day'] == 'Sun')]
print(f"Weekend meals: {len(weekend)} rows")
weekend.head()

In [ ]:
# NOT: non-smokers
non_smokers = tips[~(tips['smoker'] == 'Yes')]
# Equivalent to: tips[tips['smoker'] == 'No']
print(f"Non-smoker meals: {len(non_smokers)}")

In [ ]:
# Complex filter: Female non-smokers at dinner with a tip over $3
filtered = tips[
    (tips['sex'] == 'Female') &
    (tips['smoker'] == 'No') &
    (tips['time'] == 'Dinner') &
    (tips['tip'] > 3)
]
print(f"Matched {len(filtered)} rows")
filtered.head()

## `.isin()` — Check Membership in a List

Instead of chaining multiple `|` conditions, use `.isin()` when checking if a column value is one of several options.

In [ ]:
# Weekend days using isin (cleaner than the OR approach above)
weekend = tips[tips['day'].isin(['Sat', 'Sun'])]
weekend['day'].value_counts()

In [ ]:
# Negate with ~ to get weekdays
weekdays = tips[~tips['day'].isin(['Sat', 'Sun'])]
weekdays['day'].value_counts()

## `.between()` — Range Check

`.between(left, right)` is a convenient shorthand for `(col >= left) & (col <= right)`. Both endpoints are **inclusive** by default.

In [ ]:
# Bills between $15 and $25
mid_range = tips[tips['total_bill'].between(15, 25)]
print(f"Bills between $15 and $25: {len(mid_range)} rows")
mid_range[['total_bill', 'tip']].describe()

## `.query()` Method

The `.query()` method lets you write filter expressions as strings, which can be more readable for complex conditions. Column names are referenced directly (no `df['col']` syntax needed).

In [ ]:
# Simple query
tips.query('total_bill > 40').head()

In [ ]:
# Multiple conditions (use 'and', 'or', 'not' — the string syntax is different from bracket filtering)
tips.query('time == "Dinner" and tip > 5 and size >= 3').head()

In [ ]:
# Use @ to reference Python variables inside a query string
min_bill = 20
max_bill = 35
tips.query('@min_bill <= total_bill <= @max_bill').head()

In [ ]:
# isin equivalent in query
tips.query('day in ["Sat", "Sun"]').head()

## Assigning Filtered Results

Filtering always returns a **new** DataFrame. The original is unchanged unless you explicitly reassign.

In [ ]:
# Store the filtered result for further analysis
large_parties = tips[tips['size'] >= 4].copy()
large_parties['tip_pct'] = (large_parties['tip'] / large_parties['total_bill']) * 100
print(f"Average tip % for large parties: {large_parties['tip_pct'].mean():.1f}%")
large_parties[['total_bill', 'tip', 'tip_pct', 'size']].head()

> **Pro Tip**: Use `.copy()` when you intend to modify a filtered DataFrame. Without it you may get a `SettingWithCopyWarning` because the filtered result might be a view of the original data.

## Summary

In this notebook we covered conditional filtering in pandas:

- Comparison operators on columns produce boolean Series that act as row filters.
- Combine conditions with `&` (and), `|` (or), `~` (not) — always wrap each condition in parentheses.
- `.isin()` cleanly checks membership against a list of values.
- `.between()` provides an inclusive range check.
- `.query()` offers a string-based syntax that is often more readable for complex filters.
- Always `.copy()` a filtered result before modifying it to avoid unintended side effects.

Next up: **Useful Methods** — apply, sort, value counts, and more.